# Chapitre 3 — Importer, exporter, faire connaissance avec un jeu de données

## Le problème

Les données arrivent rarement sous forme de dictionnaire Python : elles sont dans des fichiers CSV, des classeurs Excel, du JSON, des bases SQL. La première étape de tout travail consiste à les charger **correctement** (bon séparateur, bons types, bonnes valeurs manquantes), puis à comprendre leur contenu avant de calculer quoi que ce soit.

Ce chapitre couvre :

1. la lecture et l'écriture des formats courants (CSV, Excel, JSON) ;
2. les outils d'inspection : `head`, `tail`, `info`, `describe`, `isna`, `nunique` ;
3. une méthode systématique pour découvrir un fichier inconnu.

In [27]:
import os
import numpy as np
import pandas as pd

os.makedirs("../sorties", exist_ok=True)     # dossier pour les fichiers produits par le cours

## 1. Lire un CSV

Un CSV (*comma-separated values*) est un fichier texte : une ligne par enregistrement, des valeurs séparées par un caractère. La fonction `pd.read_csv` en possède plus de cinquante paramètres ; une dizaine suffit dans la pratique.

In [9]:
clients = pd.read_csv("../donnees/demo/clients.csv")
clients.head(3) # => Seulement les 3 premières lignes
# clients # => Tout le dataframe

,client_id,prenom,ville,segment,age,date_inscription
0,101,Amina,Anvers,Particulier,40.0,2025-02-09
1,102,Lucas,Liège,Pro,51.0,2023-09-11
2,103,Sofia,Charleroi,Particulier,51.0,2025-04-05


### Quand le fichier ne suit pas les conventions

Le fichier `commandes_brutes.csv` utilise le point-virgule comme séparateur, comme beaucoup d'exports Excel en Europe.

> **Prédiction.** Que va produire `pd.read_csv` sans préciser le séparateur ? Combien de colonnes ?

In [10]:
brut = pd.read_csv("../donnees/demo/commandes_brutes.csv")
brut.shape

(243, 1)

In [11]:
brut.head(2)

,commande_id;client_id;produit_id;date_commande;quantite;prix_unitaire;canal;statut
5001;999;P08;02/01/2025 20:20;5;79,00 €;WEB;livrée
5002;105;P09;04/01/2025 19:56;1;549,00 €;WEB;livrée


L'en-tête ne contient aucune virgule : Pandas crée **une seule colonne**, dont le nom est la ligne d'en-tête entière. Les lignes de données, elles, contiennent une virgule, celle des prix (`79,00 €`) : chacune est donc découpée en deux morceaux, et le premier morceau devient l'index. Le résultat est absurde, et pourtant aucune erreur n'est levée.

Le réflexe est de vérifier `shape` et `head()` juste après chaque lecture.

In [15]:
brut = pd.read_csv("../donnees/demo/commandes_brutes.csv", sep=";")
brut.shape
brut.head(3)

,commande_id,client_id,produit_id,date_commande,quantite,prix_unitaire,canal,statut
0,5001,999,P08,02/01/2025 20:20,5,"79,00 €",WEB,livrée
1,5002,105,P09,04/01/2025 19:56,1,"549,00 €",WEB,livrée
2,5003,998,P12,06/01/2025 09:09,1,"29,90 €",WEB,annulée


### Les paramètres utiles de `read_csv`

| Paramètre | Rôle | Exemple |
|---|---|---|
| `sep` | séparateur | `sep=";"`, `sep="\t"`, `sep="\|"` |
| `index_col` | colonne à utiliser comme index | `index_col="client_id"` |
| `usecols` | ne lire que certaines colonnes | `usecols=["Name", "Age"]` |
| `nrows` | ne lire que les n premières lignes | `nrows=1000` |
| `dtype` | imposer des types | `dtype={"code_postal": "str"}` |
| `parse_dates` | convertir des colonnes en dates | `parse_dates=["date_inscription"]` |
| `na_values` | textes à considérer comme manquants | `na_values=["-", "inconnu"]` |
| `decimal` | séparateur décimal | `decimal=","` |
| `encoding` | encodage du fichier | `encoding="latin-1"` |
| `compression` | fichier compressé | `compression="zip"` (déduit de l'extension) |

In [26]:
clients = pd.read_csv(
    "../donnees/demo/clients.csv",
    index_col="client_id",
    parse_dates=["date_inscription"],
)
clients.dtypes

prenom                         str
ville                          str
segment                        str
age                        float64
date_inscription    datetime64[us]
dtype: object

In [25]:
clients.head(3)

,prenom,ville,segment,age,date_inscription
client_id,,,,,
101,Amina,Anvers,Particulier,40.0,2025-02-09
102,Lucas,Liège,Pro,51.0,2023-09-11
103,Sofia,Charleroi,Particulier,51.0,2025-04-05


L'index contient maintenant les identifiants des clients, et la date d'inscription est une vraie date (`datetime64`), sur laquelle on pourra calculer.

### Lire directement depuis une URL

`read_csv` accepte une adresse web. Les exercices du dossier `drills` utilisent cette possibilité ; des copies locales sont fournies dans `donnees/drills/` pour travailler hors ligne. Le fichier Chipotle est séparé par des tabulations, le fichier des utilisateurs par des barres verticales :

In [ ]:
chipo = pd.read_csv("../donnees/drills/chipotle.tsv", sep="\t")
users = pd.read_csv("../donnees/drills/u.user", sep="|", index_col="user_id")
chipo.shape, users.shape

## 2. Écrire un CSV

In [28]:
clients.to_csv("../sorties/clients_export.csv")
open("../sorties/clients_export.csv", encoding="utf-8").read()[:180]

'client_id,prenom,ville,segment,age,date_inscription\n101,Amina,Anvers,Particulier,40.0,2025-02-09\n102,Lucas,Liège,Pro,51.0,2023-09-11\n103,Sofia,Charleroi,Particulier,51.0,2025-04-05'

Par défaut, `to_csv` écrit aussi l'index. Avec un index sans signification (0, 1, 2…), on ajoute `index=False`, sinon la relecture crée une colonne `Unnamed: 0`.

In [ ]:
produits = pd.read_csv("../donnees/demo/produits.csv")
produits.to_csv("../sorties/produits_avec_index.csv")
# produits.to_csv("../sorties/produits_avec_index2.csv", index_label="hello")
pd.read_csv("../sorties/produits_avec_index.csv").head(2)

,Unnamed: 0,produit_id,libelle,categorie,prix
0,0,P01,Clavier mécanique,Périphérique,89.9
1,1,P02,Souris sans fil,Périphérique,24.5


In [31]:
produits.to_csv("../sorties/produits.csv", index=False)
pd.read_csv("../sorties/produits.csv").head(2)

,produit_id,libelle,categorie,prix
0,P01,Clavier mécanique,Périphérique,89.9
1,P02,Souris sans fil,Périphérique,24.5


La colonne `Unnamed: 0` apparaît d'ailleurs dans le classeur Sakila de l'exercice du chapitre 5 : quelqu'un a exporté un DataFrame sans `index=False`.

Pour un fichier volumineux, la compression se déduit de l'extension :

In [34]:
produits.to_csv("../sorties/produits.csv.zip", index=False)
pd.read_csv("../sorties/produits.csv.zip").shape

(15, 4)

## 3. Excel

La lecture et l'écriture des fichiers `.xlsx` passent par la bibliothèque `openpyxl`, installée avec le projet.

In [35]:
xl = pd.ExcelFile("../donnees/bstorm/sakila.xlsx")
xl.sheet_names

['actor', 'film', 'category', 'film_actor', 'film_category']

In [ ]:
actor = pd.read_excel("../donnees/bstorm/sakila.xlsx", sheet_name="actor")
actor.head(3)

,Unnamed: 0,actor_id,first_name,last_name,last_update
0,0,1,PENELOPE,GUINESS,2006-02-15 04:34:33
1,1,2,NICK,WAHLBERG,2006-02-15 04:34:33
2,2,3,ED,CHASE,2006-02-15 04:34:33


`sheet_name=None` charge **toutes** les feuilles dans un dictionnaire `{nom_de_feuille: DataFrame}`.

In [ ]:
feuilles = pd.read_excel("../donnees/bstorm/sakila.xlsx", sheet_name=None)
{nom: f.shape for nom, f in feuilles.items()}

{'actor': (200, 5),
 'film': (1000, 14),
 'category': (16, 4),
 'film_actor': (5462, 4),
 'film_category': (1000, 4)}

Pour écrire plusieurs feuilles dans un même classeur, on utilise un `ExcelWriter` dans un bloc `with`, qui ferme correctement le fichier.

In [38]:
with pd.ExcelWriter("../sorties/boutique.xlsx") as writer:
    clients.to_excel(writer, sheet_name="clients")
    produits.to_excel(writer, sheet_name="produits", index=False)

pd.read_excel("../sorties/boutique.xlsx", sheet_name="produits", usecols=["produit_id", "prix"]).head(3)

,produit_id,prix
0,P01,89.9
1,P02,24.5
2,P03,249.0


## 4. JSON

Le JSON est le format des API web. Deux formes reviennent souvent :

- une **liste d'enregistrements** : `[{"col": val, ...}, {...}]` ;
- un **dictionnaire de colonnes** : `{"col": {index: val, ...}, ...}`.

Le paramètre `orient` choisit la forme à l'écriture comme à la lecture.

In [39]:
produits.head(2).to_json(orient="records", force_ascii=False)

'[{"produit_id":"P01","libelle":"Clavier mécanique","categorie":"Périphérique","prix":89.9},{"produit_id":"P02","libelle":"Souris sans fil","categorie":"Périphérique","prix":24.5}]'

In [40]:
produits.head(2).to_json(orient="columns", force_ascii=False)

'{"produit_id":{"0":"P01","1":"P02"},"libelle":{"0":"Clavier mécanique","1":"Souris sans fil"},"categorie":{"0":"Périphérique","1":"Périphérique"},"prix":{"0":89.9,"1":24.5}}'

In [41]:
from io import StringIO
texte = '[{"produit_id": "P01", "prix": 89.9}, {"produit_id": "P02", "prix": 24.5}]'
pd.read_json(StringIO(texte))

,produit_id,prix
0,P01,89.9
1,P02,24.5


`read_json` attend un fichier, une URL ou un objet fichier : le texte brut doit être enveloppé dans `StringIO`. Quand le JSON contient des objets imbriqués, `pd.json_normalize` les aplatit en colonnes (chapitre 9 et chapitre 11).

## 5. Faire connaissance avec un jeu de données

Chargeons le jeu de données de l'exercice principal : 120 ans de Jeux olympiques.

In [42]:
jo = pd.read_csv("../donnees/bstorm/Exercices_Pandas_athlete_events.csv")
jo.shape

(271116, 15)

### Regarder les données

In [43]:
jo.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


In [44]:
jo.tail(3)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
271113,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Team",NaN
271114,135571,Tomasz Ireneusz ya,M,30.0,185.0,96.0,Poland,POL,1998 Winter,1998,Winter,Nagano,Bobsleigh,Bobsleigh Men's Four,NaN
271115,135571,Tomasz Ireneusz ya,M,34.0,185.0,96.0,Poland,POL,2002 Winter,2002,Winter,Salt Lake City,Bobsleigh,Bobsleigh Men's Four,NaN


In [62]:
jo.sample(3)     # trois lignes au hasard : souvent plus représentatif que head
# jo.sample(3, random_state=1)     # Random state = seed

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
43626,22496,Dario Alonzo Cologna,M,27.0,179.0,74.0,Switzerland,SUI,2014 Winter,2014,Winter,Sochi,Cross Country Skiing,Cross Country Skiing Men's 30 km Skiathlon,Gold
214199,107550,George Schroeder,M,NaN,NaN,NaN,United States,USA,1904 Summer,1904,Summer,St. Louis,Gymnastics,Gymnastics Men's Individual All-Around,NaN
200126,100513,"Vincent ""Vinnie"" Richards",M,21.0,178.0,NaN,United States-2,USA,1924 Summer,1924,Summer,Paris,Tennis,Tennis Men's Doubles,Gold


### La structure : `info()`

`info()` résume en une commande : le nombre de lignes, le type de chaque colonne, le nombre de valeurs **non nulles**, la mémoire occupée.

In [63]:
jo.info()

<class 'pandas.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  str    
 2   Sex     271116 non-null  str    
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  str    
 7   NOC     271116 non-null  str    
 8   Games   271116 non-null  str    
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  str    
 11  City    271116 non-null  str    
 12  Sport   271116 non-null  str    
 13  Event   271116 non-null  str    
 14  Medal   39783 non-null   str    
dtypes: float64(3), int64(2), str(10)
memory usage: 56.6 MB


> **Mini-problème.** À partir de la sortie de `info()` uniquement, relevez trois informations sur ce jeu de données qu'il faudra prendre en compte avant d'en tirer des conclusions.

### Les valeurs manquantes

In [64]:
jo.isna().sum()

ID             0
Name           0
Sex            0
Age         9474
Height     60171
Weight     62875
Team           0
NOC            0
Games          0
Year           0
Season         0
City           0
Sport          0
Event          0
Medal     231333
dtype: int64

In [65]:
(jo.isna().mean() * 100).round(1)      # en pourcentage : la moyenne d'une colonne de booléens

ID         0.0
Name       0.0
Sex        0.0
Age        3.5
Height    22.2
Weight    23.2
Team       0.0
NOC        0.0
Games      0.0
Year       0.0
Season     0.0
City       0.0
Sport      0.0
Event      0.0
Medal     85.3
dtype: float64

Une valeur manquante n'est pas toujours une anomalie. Dans `Medal`, `NaN` signifie « pas de médaille » : c'est une information. Dans `Age`, c'est une donnée inconnue.

### Les statistiques : `describe()`

In [66]:
jo.describe()

,ID,Age,Height,Weight,Year
count,271116.000000,261642.000000,210945.000000,208241.000000,271116.000000
mean,68248.954396,25.556898,175.338970,70.702393,1978.378480
std,39022.286345,6.393561,10.518462,14.348020,29.877632
min,1.000000,10.000000,127.000000,25.000000,1896.000000
25%,34643.000000,21.000000,168.000000,60.000000,1960.000000
50%,68205.000000,24.000000,175.000000,70.000000,1988.000000
75%,102097.250000,28.000000,183.000000,79.000000,2002.000000
max,135571.000000,97.000000,226.000000,214.000000,2016.000000


Par défaut, `describe()` ne résume que les colonnes numériques. `include="all"` ajoute les colonnes texte, avec le nombre de valeurs distinctes (`unique`), la plus fréquente (`top`) et sa fréquence (`freq`).

In [67]:
jo.describe(include="all")

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
count,271116.000000,271116,271116,261642.000000,210945.000000,208241.000000,271116,271116,271116,271116.000000,271116,271116,271116,271116,39783
unique,NaN,134732,2,NaN,NaN,NaN,1184,230,51,NaN,2,42,66,765,3
top,NaN,Robert Tait McKenzie,M,NaN,NaN,NaN,United States,USA,2000 Summer,NaN,Summer,London,Athletics,Football Men's Football,Gold
freq,NaN,58,196594,NaN,NaN,NaN,17847,18853,13821,NaN,222552,22426,38624,5733,13372
mean,68248.954396,NaN,NaN,25.556898,175.338970,70.702393,NaN,NaN,NaN,1978.378480,NaN,NaN,NaN,NaN,NaN
std,39022.286345,NaN,NaN,6.393561,10.518462,14.348020,NaN,NaN,NaN,29.877632,NaN,NaN,NaN,NaN,NaN
min,1.000000,NaN,NaN,10.000000,127.000000,25.000000,NaN,NaN,NaN,1896.000000,NaN,NaN,NaN,NaN,NaN
25%,34643.000000,NaN,NaN,21.000000,168.000000,60.000000,NaN,NaN,NaN,1960.000000,NaN,NaN,NaN,NaN,NaN
50%,68205.000000,NaN,NaN,24.000000,175.000000,70.000000,NaN,NaN,NaN,1988.000000,NaN,NaN,NaN,NaN,NaN
75%,102097.250000,NaN,NaN,28.000000,183.000000,79.000000,NaN,NaN,NaN,2002.000000,NaN,NaN,NaN,NaN,NaN


On peut aussi cibler un type de colonnes avec `select_dtypes`.

In [68]:
jo.select_dtypes(include="number").columns.tolist(), jo.select_dtypes(include="str").columns.tolist()

(['ID', 'Age', 'Height', 'Weight', 'Year'],
 ['Name',
  'Sex',
  'Team',
  'NOC',
  'Games',
  'Season',
  'City',
  'Sport',
  'Event',
  'Medal'])

### Le nombre de valeurs distinctes

In [69]:
jo.nunique()

ID        135571
Name      134732
Sex            2
Age           74
Height        95
Weight       220
Team        1184
NOC          230
Games         51
Year          35
Season         2
City          42
Sport         66
Event        765
Medal          3
dtype: int64

> **Pourquoi ?** La colonne `ID` compte 135 571 valeurs distinctes pour 271 116 lignes. Qu'est-ce que cela indique sur ce qu'est une ligne de ce fichier ?

### Une fonction d'inspection réutilisable

La correction Bstorm de l'exercice JO propose de regrouper ces informations dans un seul tableau. C'est une bonne habitude : on écrit cette fonction une fois, on la réutilise sur chaque nouveau fichier.

In [70]:
def info(df):
    """Résumé d'un DataFrame : type, valeurs manquantes, valeurs distinctes."""
    return pd.DataFrame({
        "type": df.dtypes,
        "non_nuls": df.count(),
        "manquants": df.isna().sum(),
        "pct_manquants": (df.isna().mean() * 100).round(1),
        "distincts": df.nunique(),
    })

info(jo)

,type,non_nuls,manquants,pct_manquants,distincts
ID,int64,271116,0,0.0,135571
Name,str,271116,0,0.0,134732
Sex,str,271116,0,0.0,2
Age,float64,261642,9474,3.5,74
Height,float64,210945,60171,22.2,95
Weight,float64,208241,62875,23.2,220
Team,str,271116,0,0.0,1184
NOC,str,271116,0,0.0,230
Games,str,271116,0,0.0,51
Year,int64,271116,0,0.0,35


### La mémoire

In [71]:
print(round(jo.memory_usage(deep=True).sum() / 1e6, 1), "Mo en mémoire")

59.4 Mo en mémoire


Le fichier fait 41 Mo sur disque mais le DataFrame en occupe davantage. Les colonnes texte et leurs millions de chaînes en sont la cause. Une conversion des colonnes répétitives (`Sex`, `Season`, `NOC`, `Medal`…) en `category` réduit fortement cette taille (chapitre 5).

## Une méthode pour découvrir un fichier

1. **Lire** et vérifier immédiatement `shape` et `head()` (séparateur, en-tête, encodage).
2. **Structure** : `info()` pour les types et les non-nuls.
3. **Manquants** : `isna().sum()`, et se demander ce que signifie chaque `NaN`.
4. **Distributions** : `describe()` et `describe(include="all")`.
5. **Cardinalités** : `nunique()` pour repérer identifiants, catégories, textes libres.
6. **Granularité** : qu'est-ce qu'une ligne ? (un athlète ? une participation ? une épreuve ?)

## Pièges fréquents

| Situation | Symptôme | Réflexe |
|---|---|---|
| Mauvais séparateur | une seule colonne | `sep=` et vérifier `shape` |
| Index exporté par erreur | colonne `Unnamed: 0` | `index=False` à l'écriture, ou `index_col=0` à la lecture |
| Dates lues comme du texte | type `str` | `parse_dates=` ou `pd.to_datetime` |
| Codes postaux ou identifiants à zéros initiaux | `0123` devient `123` | `dtype={"col": "str"}` |
| Décimales avec une virgule | colonne lue en texte | `decimal=","` |
| `describe()` sans les colonnes texte | colonnes absentes du résumé | `include="all"` |

## Exercices

- Exercice Bstorm **JO**, questions **1 à 8** (`exercices/bstorm/01_pandas_JO/`). Les données sont aussi dans `donnees/bstorm/`.
- Renforcement : drill **D01 Occupation**, puis **D01 Chipotle** étapes 1 à 12 (`exercices/drills/`).

## À retenir

- `read_csv`, `read_excel`, `read_json` et leurs symétriques `to_csv`, `to_excel`, `to_json`.
- Toujours vérifier `shape` et `head()` après une lecture : une erreur de lecture est souvent silencieuse.
- `info()`, `isna().sum()`, `describe(include="all")`, `nunique()` : quatre commandes pour comprendre un fichier.
- Une valeur manquante peut être une absence d'information… ou une information.
- Identifier la granularité d'une ligne avant tout calcul.